# Fine-Tuning LLaMA 1B with LoRA for News Summarization

This project fine-tunes a 1B-parameter LLaMA model using Low-Rank Adaptation (LoRA) on a subset of the CNN/DailyMail dataset for abstractive summarization.

The goal is to train the model to learn summarization patterns by predicting summary text conditioned on full news articles.This project fine-tunes a 1B-parameter LLaMA model for abstractive news summarization using the CNN/DailyMail dataset. The model will learn to generate concise summaries by predicting the next token conditioned on a news article prompt.

## Importing Modules

In [1]:
!pip install transformers datasets accelerate peft

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/tobi/anaconda3/envs/llm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [39]:
from peft import LoraConfig, get_peft_model

## Dataset Selection

The CNN/DailyMail dataset consists of full news articles paired with human-written summaries.

Each training example contains:
- A long news article
- A concise summary (highlights)
- An id

This structure makes it ideal for learning summarization patterns, where the model must compress detailed information into shorter structured outputs.

Due to computational constraints, I fine-tuned on a subset of 150,000 training examples rather than using all training examples.

In [34]:
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:150000]")

print(dataset)
print(dataset[0])

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 150000
})
{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Pa

In [35]:
print(len(dataset))

150000


## How the Model Learns Patterns and Completions

The model is trained on sequences formatted as:

summarize: \<article\>

\### Summary:
\<summary\>

During training, the model repeatedly observes that after the prompt "### Summary:" follows a concise description of the article's main event, key entities, and outcome.

Because LLaMA is a causal language model, it learns by predicting the next token in the sequence. Over thousands of examples, it learns structural patterns such as:

- Identifying the primary event in a news article
- Preserving important named entities
- Removing minor or repeated details
- Writing in a neutral journalistic tone
- Compressing long narratives into short structured summaries

After training, when given a new article with the same prompt format, the model generates a summary by using learned summarization pattern from the train data.

In [ ]:
model_name = "meta-llama/Llama-3.2-1B"  

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [21]:
tokenizer.pad_token = tokenizer.eos_token

### Prompt Formatting

Each example is converted into a prompt-completion format so the language model can learn the summarization task.


In [36]:
def format_example(example):
    prompt = f"""summarize: {example['article']}

### Summary:
{example['highlights']}"""
    
    return {"text": prompt}

formatted_dataset = dataset.map(format_example)

Map: 100%|██████████| 150000/150000 [00:06<00:00, 24389.35 examples/s]


In [37]:
formatted_dataset

Dataset({
    features: ['article', 'highlights', 'id', 'text'],
    num_rows: 150000
})

In [38]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    
    tokens["labels"] = tokens["input_ids"].copy()
    
    return tokens

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 150000/150000 [00:39<00:00, 3759.33 examples/s]


### Low-Rank Adaptation (LoRA)

Instead of fine-tuning all parameters of the LLaMA model, i use LoRA to train only a small set of adapter weights. This significantly reduces memory usage and training time while still allowing the model to learn the summarization task.

In [ ]:
lora_config = LoraConfig(
    r=8,                       
    lora_alpha=16,             
    target_modules=["q_proj", "v_proj"],  
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [41]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


/home/tobi/anaconda3/envs/llm_env/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/tobi/anaconda3/envs/llm_env/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [42]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama-lora-summarizer",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=500,
    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

In [43]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

/tmp/ipykernel_49546/2100966235.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [44]:
trainer.train()

Step,Training Loss
50,2.349000
100,2.244600
150,2.225400
200,2.212500
250,2.211900
300,2.173300
350,2.215900
400,2.187600
450,2.172600
500,2.193000


TrainOutput(global_step=18750, training_loss=2.1759292720540366, metrics={'train_runtime': 14974.1242, 'train_samples_per_second': 10.017, 'train_steps_per_second': 1.252, 'total_flos': 4.48818315264e+17, 'train_loss': 2.1759292720540366, 'epoch': 1.0})

In [14]:
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Using CPU")

NVIDIA GeForce RTX 4080 Laptop GPU


In [49]:
article = dataset[0]["article"]

In [50]:
def model_pred(article):
    model.eval()

    prompt = f"""summarize: {article}

    ### Summary:
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9
    )

    return(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [51]:
prediction = model_pred(article)
print(prediction)

summarize: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Det

In [54]:
print(dataset[0]["highlights"])

Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [47]:
model.save_pretrained("./llama-lora-summarizer")
tokenizer.save_pretrained("./llama-lora-summarizer")

('./llama-lora-summarizer/tokenizer_config.json',
 './llama-lora-summarizer/special_tokens_map.json',
 './llama-lora-summarizer/tokenizer.json')

In [48]:
test_dataset = load_dataset("cnn_dailymail", "3.0.0", split="test[:150000]")

print(test_dataset)
print(test_dataset[0])

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 11490
})
{'article': '(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join

In [52]:
article = test_dataset[0]["article"]

prediction = model_pred(article)
print(prediction)

summarize: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday's c

In [53]:
print(test_dataset[0]["highlights"])

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .
